## Online Retail Transactions

**Understanding the Dataset**

The online retail csv file is an extensive collection of data relating to ecommerce transactions. This dataset provides a detailed view of sales activities within the online retail sector, covering numerous essential attributes necessary for a quantitative understanding of consumer behavior and the overall business performance.

**The Columns**:
* **InvoiceNo**	A unique identification number assigned to each transaction. (Numeric)
* **StockCode**	A unique identification code assigned to each product sold by the retailer. (Numeric)
* **Description**	A brief description of the product sold. (Text)
* **Quantity**	The number of units of the product sold in each transaction. (Numeric)
* **InvoiceDate**	The exact date and time when the transaction occurred. (Date/Time)
* **UnitPrice**	The price per unit of the product sold. (Numeric)
* **Country**	The country where the customer resides. (Text)

Source: https://www.kaggle.com/datasets/thedevastator/online-retail-transaction-records

**Import Pyspark and start a SparkSession**

In [40]:
from pyspark.sql import SparkSession

In [41]:
# start a spark session
spark = (SparkSession.builder
         .master('local')
         .appName("online_retail_project")
         .getOrCreate())

# create a spark context
sc = spark.sparkContext

**Step 1: Load the csv file** 

In [13]:
# load data
data = sc.textFile('dataset/retail_dataset.csv')

In [43]:
# let's re-read the dataset using spark's csv reader to make sure all columns are correctly infered.
df = spark.read.csv('dataset/retail_dataset.csv', header=True, inferSchema=True)
data = df.rdd
print(type(data))

<class 'pyspark.rdd.RDD'>


confirmed that the data is of type rdd, that means our data was read successfully.

**Step 2: Cleaning Data**

Now let's get a sense of our dataset and possible data quality issues. Now, our RDD is full of a collection of row objects

**1. Row count**

In [47]:
# check count
print(f'There are {data.count()} of data')

There are 541909 of data


**2. Check columns**

In [49]:
# let's check the count of missing or null values in each column
missing_counts = data.map(lambda row: [(col, 1) if row[col] is None else (col, 0) for col in row.asDict()]) \
    .flatMap(lambda x: x) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()

so we create a tuple containing the column and the number of rows, and we convert it as a dictionary so we can iterate over it, after we flatten each tuple, for for each flattened tuple we perform an aggreagation using reducebykey.

In [52]:
for col in missing_counts:
    print(col)

('index', 0)
('InvoiceNo', 0)
('StockCode', 0)
('Description', 1454)
('Quantity', 0)
('InvoiceDate', 0)
('UnitPrice', 0)
('CustomerID', 135080)
('Country', 0)


* We can see that there are 1454 missing values in the Description.
* 135k rows of data also have missing customer ids

**3. Preview data** - Check data distribution

In [61]:
# show a sample of the description column
data.map(lambda row: (row['Description'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[('WHITE HANGING HEART T-LIGHT HOLDER', 2369),
 ('WHITE METAL LANTERN', 328),
 ('CREAM CUPID HEARTS COAT HANGER', 293),
 ('KNITTED UNION FLAG HOT WATER BOTTLE', 473),
 ('RED WOOLLY HOTTIE WHITE HEART.', 449),
 ('SET 7 BABUSHKA NESTING BOXES', 389),
 ('GLASS STAR FROSTED T-LIGHT HOLDER', 141),
 ('HAND WARMER UNION JACK', 515),
 ('HAND WARMER RED POLKA DOT', 18),
 ('ASSORTED COLOUR BIRD ORNAMENT', 1501)]

In [60]:
# preview the customer id column
data.map(lambda row: (row['CustomerID'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[(17850.0, 312),
 (13047.0, 196),
 (12583.0, 251),
 (13748.0, 28),
 (15100.0, 6),
 (15291.0, 109),
 (14688.0, 359),
 (17809.0, 64),
 (15311.0, 2491),
 (14527.0, 1011)]

In [62]:
# preview Country data. 
data.map(lambda row: (row['Country'], 1)).reduceByKey(lambda a, b: a + b).take(10)

[('United Kingdom', 495478),
 ('France', 8557),
 ('Australia', 1259),
 ('Netherlands', 2371),
 ('Germany', 9495),
 ('Norway', 1086),
 ('EIRE', 8196),
 ('Switzerland', 2002),
 ('Spain', 2533),
 ('Poland', 341)]

In [64]:
# view missing values in CustomerID column
data.filter(lambda row: row['CustomerID'] is None).take(10)

[Row(index=622, InvoiceNo='536414', StockCode='22139', Description=None, Quantity=56, InvoiceDate='12/1/2010 11:52', UnitPrice=0.0, CustomerID=None, Country='United Kingdom'),
 Row(index=1443, InvoiceNo='536544', StockCode='21773', Description='DECORATIVE ROSE BATHROOM BOTTLE', Quantity=1, InvoiceDate='12/1/2010 14:32', UnitPrice=2.51, CustomerID=None, Country='United Kingdom'),
 Row(index=1444, InvoiceNo='536544', StockCode='21774', Description='DECORATIVE CATS BATHROOM BOTTLE', Quantity=2, InvoiceDate='12/1/2010 14:32', UnitPrice=2.51, CustomerID=None, Country='United Kingdom'),
 Row(index=1445, InvoiceNo='536544', StockCode='21786', Description='POLKADOT RAIN HAT ', Quantity=4, InvoiceDate='12/1/2010 14:32', UnitPrice=0.85, CustomerID=None, Country='United Kingdom'),
 Row(index=1446, InvoiceNo='536544', StockCode='21787', Description='RAIN PONCHO RETROSPOT', Quantity=2, InvoiceDate='12/1/2010 14:32', UnitPrice=1.66, CustomerID=None, Country='United Kingdom'),
 Row(index=1447, Invoic

**4. Replace missing values**

From the exploration, we can see that the required columns for KPI analysis (sales) is filled, and there are no nulls. Therefore dropping **null** customerIDs or Descriptions would remove these vital data points.

In [69]:
# replace all missing values in customer id

# because we created the dataframe before converting to rdd. the elements are spark row objects, and executors need to know what Row is when creating or modifying them. Without the import, they throw a NameError
from pyspark.sql import Row 

# extract the columns... convert the rdd to df to extract columns
columns = data.toDF().columns

updated_data = data.map(lambda row: Row(**
                                       {
                                           col: ("Unknown" if col == "CustomerID" and row[col] is None else row[col])
                                           for col in columns
                                       }))

In [72]:
updated_data.filter(lambda row: row['CustomerID'] is None).take(10)

[]

There are no nulls in the column now

**Step 3: Analysis**

**1. Sales Overview**

In [75]:
# Total number of transactions (Unique)

total_transactions = updated_data.map(lambda row: row['InvoiceNo']).distinct().count()

print(f"There are {total_transactions} distinct number of transactions")

There are 25900 distinct number of transactions


In [76]:
# Total Quantity Sold

total_quantity = updated_data.map(lambda row: row['Quantity']).sum()

print(f"There a total of {total_quantity} of items sold")

There a total of 5176450 of items sold


In [85]:
# Total Revenue

total_revenue = updated_data.map(lambda row: row['UnitPrice'] * row['Quantity']).sum()

print(f"The total revenue made is {round(total_revenue, 2)}")

The total revenue made is 9747747.93


In [87]:
# Average money generated per transaction

print(f"The average order value for each transaction is: {round(total_revenue / total_transactions, 2)}")

The average order value for each transaction is: 376.36


**2. Product Insights**

In [93]:
# Top selling products by quantity

top_products_qty = (updated_data.map(lambda row: (row['Description'], row['Quantity']))
                    .reduceByKey(lambda a, b: a + b)
                    .sortBy(lambda x: x[1], ascending=False)
                    .take(10)
                   )
print("These are the top 10 products by quantity sold: \n")
for prd in top_products_qty:
    print(prd)

These are the top 10 products by quantity sold: 

('WORLD WAR 2 GLIDERS ASSTD DESIGNS', 53847)
('JUMBO BAG RED RETROSPOT', 47363)
('ASSORTED COLOUR BIRD ORNAMENT', 36381)
('POPCORN HOLDER', 36334)
('PACK OF 72 RETROSPOT CAKE CASES', 36039)
('WHITE HANGING HEART T-LIGHT HOLDER', 35317)
('RABBIT NIGHT LIGHT', 30680)
('MINI PAINT SET VINTAGE ', 26437)
('PACK OF 12 LONDON TISSUES ', 26315)
('PACK OF 60 PINK PAISLEY CAKE CASES', 24753)


In [109]:
# Top products by revenue
top_products_rev = (
        updated_data.map(lambda row: (row['Description'], row['Quantity'] * row['UnitPrice']))
       .reduceByKey(lambda a, b: round(a + b, 2))
       .sortBy(lambda x: x[1], ascending=False)
       .take(10)
)


print("These are the top 10 products by revenue generated: \n")
for prd in top_products_rev:
    print(prd)
    

These are the top 10 products by revenue generated: 

('DOTCOM POSTAGE', 206245.48)
('REGENCY CAKESTAND 3 TIER', 164762.19)
('WHITE HANGING HEART T-LIGHT HOLDER', 99668.47)
('PARTY BUNTING', 98302.98)
('JUMBO BAG RED RETROSPOT', 92356.03)
('RABBIT NIGHT LIGHT', 66756.59)
('POSTAGE', 66230.64)
("PAPER CHAIN KIT 50'S CHRISTMAS ", 63791.94)
('ASSORTED COLOUR BIRD ORNAMENT', 58959.73)
('CHILLI LIGHTS', 53768.06)


**3. Customer Insights**

In [110]:
# Top customers who spend the most - top customers by total spending
top_spending_customers = (
    updated_data.filter(lambda row: row['CustomerID'] != "Unknown")
    .map(lambda row: (row['CustomerID'], row['Quantity'] * row['UnitPrice']))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortBy(lambda x: x[1], ascending=False)
    .take(10)
)

print("These are the top spending customers by their IDs \n")
for element in top_spending_customers:
    print(element)

These are the top spending customers by their IDs 

(14646.0, 279489.02)
(18102.0, 256438.49)
(17450.0, 187482.17)
(14911.0, 132572.62)
(12415.0, 123725.45)
(14156.0, 113384.14)
(17511.0, 88125.38)
(16684.0, 65892.08)
(13694.0, 62653.1)
(15311.0, 59419.34)


In [112]:
# Top customers by purchase frequency 

top_customers_frequency = (
    updated_data.filter(lambda row: row['CustomerID'] != "Unknown")
       .map(lambda row: (row['CustomerID'], 1))
       .reduceByKey(lambda a, b: round(a + b, 2))
       .sortBy(lambda x: x[1], ascending=False)
       .take(10)
)

print("These are the top customers with highpurchase frequency \n")
for element in top_customers_frequency:
    print(element)

These are the top customers with highpurchase frequency 

(17841.0, 7983)
(14911.0, 5903)
(14096.0, 5128)
(12748.0, 4642)
(14606.0, 2782)
(15311.0, 2491)
(14646.0, 2085)
(13089.0, 1857)
(13263.0, 1677)
(14298.0, 1640)


In [122]:
# How much do customers spend on average per order

avg_order_value_per_customer = (
    updated_data.filter(lambda row: row['CustomerID'] != "Unknown")
    .map(lambda row: (row['CustomerID'], (row['Quantity'] * row['UnitPrice'], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .mapValues(lambda x: round(x[0] / x[1], 2))
    .sortBy(lambda x: x[1], ascending=False)
    .take(10)
)

for element in avg_order_value_per_customer:
    print(element)

(15195.0, 3861.0)
(13135.0, 3096.0)
(17846.0, 2033.1)
(16532.0, 1687.2)
(15749.0, 1435.73)
(16000.0, 1377.08)
(16754.0, 1001.2)
(12798.0, 872.13)
(17553.0, 743.8)
(17949.0, 667.73)


**4. Time Analysis**

In [123]:
# Sales over time
from datetime import datetime

# let's create a date parsing function to parse InvoiceDate
def parse_date(row):
    dt = datetime.strptime(row['InvoiceDate'], "%m/%d/%Y %H:%M")
    return (row['InvoiceNo'], row['StockCode'], row['Description'], row['Quantity'],
            dt, row['UnitPrice'], row['CustomerID'], row['Country'])


In [124]:
# parse the rdd

parsed_data = updated_data.map(lambda row: Row(
    InvoiceNo=row['InvoiceNo'],
    StockCode=row['StockCode'],
    Description=row['Description'],
    Quantity=row['Quantity'],
    InvoiceDate=datetime.strptime(row['InvoiceDate'], "%m/%d/%Y %H:%M"),
    UnitPrice=row['UnitPrice'],
    CustomerID=row['CustomerID'],
    Country=row['Country']
))

In [126]:
# Sales by Month 
sales_by_month = (
    parsed_data
    .map(lambda row: ((row.InvoiceDate.year, row.InvoiceDate.month), row.Quantity * row.UnitPrice))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortByKey()
    .collect()
)

for item in sales_by_month:
    print(item)


((2010, 12), 748957.02)
((2011, 1), 560000.26)
((2011, 2), 498062.65)
((2011, 3), 683267.08)
((2011, 4), 493207.12)
((2011, 5), 723333.51)
((2011, 6), 691123.12)
((2011, 7), 681300.11)
((2011, 8), 682680.51)
((2011, 9), 1019687.62)
((2011, 10), 1070704.67)
((2011, 11), 1461756.25)
((2011, 12), 433668.01)


In [128]:
# Sales by Day of the week

sales_by_day = (
    parsed_data
    .map(lambda row: (row.InvoiceDate.date(), row.Quantity * row.UnitPrice))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortByKey()
    .collect()
)

for item in sales_by_day:
    print(item)

(datetime.date(2010, 12, 1), 58635.56)
(datetime.date(2010, 12, 2), 46207.28)
(datetime.date(2010, 12, 3), 45620.46)
(datetime.date(2010, 12, 5), 31383.95)
(datetime.date(2010, 12, 6), 53860.18)
(datetime.date(2010, 12, 7), 45059.05)
(datetime.date(2010, 12, 8), 44189.84)
(datetime.date(2010, 12, 9), 52532.13)
(datetime.date(2010, 12, 10), 57404.91)
(datetime.date(2010, 12, 12), 17240.92)
(datetime.date(2010, 12, 13), 35379.34)
(datetime.date(2010, 12, 14), 42843.29)
(datetime.date(2010, 12, 15), 29443.69)
(datetime.date(2010, 12, 16), 48334.35)
(datetime.date(2010, 12, 17), 43534.19)
(datetime.date(2010, 12, 19), 7517.31)
(datetime.date(2010, 12, 20), 24741.75)
(datetime.date(2010, 12, 21), 47097.94)
(datetime.date(2010, 12, 22), 6134.57)
(datetime.date(2010, 12, 23), 11796.31)
(datetime.date(2011, 1, 4), 14950.48)
(datetime.date(2011, 1, 5), -1566.23)
(datetime.date(2011, 1, 6), 37392.74)
(datetime.date(2011, 1, 7), 27233.14)
(datetime.date(2011, 1, 9), 15710.8)
(datetime.date(2011, 

In [130]:
# Sales By Hour of the Day
sales_by_hour = (
    parsed_data
    .map(lambda row: (row.InvoiceDate.hour, row.Quantity * row.UnitPrice))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortByKey()
    .collect()
)

for item in sales_by_hour:
    print(item)

(6, -497.35)
(7, 31009.32)
(8, 281840.86)
(9, 766734.05)
(10, 1329056.52)
(11, 1147437.92)
(12, 1362484.29)
(13, 1177506.37)
(14, 1095212.9)
(15, 1189458.28)
(16, 729140.82)
(17, 435444.11)
(18, 140574.48)
(19, 46324.99)
(20, 16020.37)


**5. Country-level Insights**

In [131]:
# Total sales by country
sales_by_country = (
    parsed_data
    .map(lambda row: (row.Country, row.Quantity * row.UnitPrice))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

for item in sales_by_country:
    print(item)

('United Kingdom', 8187806.36)
('Netherlands', 284661.54)
('EIRE', 263276.82)
('Germany', 221698.21)
('France', 197403.9)
('Australia', 137077.27)
('Switzerland', 56385.35)
('Spain', 54774.58)
('Belgium', 40910.96)
('Sweden', 36595.91)
('Japan', 35340.62)
('Norway', 35163.46)
('Portugal', 29367.02)
('Finland', 22326.74)
('Channel Islands', 20086.29)
('Denmark', 18768.14)
('Italy', 16890.51)
('Cyprus', 12946.29)
('Austria', 10154.32)
('Hong Kong', 10117.04)
('Singapore', 9120.39)
('Israel', 7907.82)
('Poland', 7213.14)
('Unspecified', 4749.79)
('Greece', 4710.52)
('Iceland', 4310.0)
('Canada', 3666.38)
('Malta', 2505.47)
('United Arab Emirates', 1902.28)
('USA', 1730.92)
('Lebanon', 1693.88)
('Lithuania', 1661.06)
('European Community', 1291.75)
('Brazil', 1143.6)
('RSA', 1002.31)
('Czech Republic', 707.72)
('Bahrain', 548.4)
('Saudi Arabia', 131.17)


In [133]:
# Number of transactions per country
transactions_by_country = (
    parsed_data
    .map(lambda row: (row.Country, row.InvoiceNo))
    .distinct()
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a, b: round(a + b, 2))
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

for item in transactions_by_country:
    print(item)

('United Kingdom', 23494)
('Germany', 603)
('France', 461)
('EIRE', 360)
('Belgium', 119)
('Spain', 105)
('Netherlands', 101)
('Switzerland', 74)
('Portugal', 71)
('Australia', 69)
('Italy', 55)
('Finland', 48)
('Sweden', 46)
('Norway', 40)
('Channel Islands', 33)
('Japan', 28)
('Poland', 24)
('Denmark', 21)
('Cyprus', 20)
('Austria', 19)
('Hong Kong', 15)
('Unspecified', 13)
('Singapore', 10)
('Malta', 10)
('Israel', 9)
('Iceland', 7)
('USA', 7)
('Greece', 6)
('Canada', 6)
('Czech Republic', 5)
('European Community', 5)
('Lithuania', 4)
('Bahrain', 4)
('United Arab Emirates', 3)
('Saudi Arabia', 2)
('Lebanon', 1)
('Brazil', 1)
('RSA', 1)


<hr>

**Convert code to pipeline script**

I converted the code into an ETL pipeline script, and i ran the script in the cell. You can see the output below. 

In [135]:
import logging
import sys
from datetime import datetime
from typing import List, Tuple, Dict, Any
from pyspark.sql import SparkSession, Row
from pyspark.rdd import RDD

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_pipeline.log'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)


class RetailETLPipeline:
    """Main ETL pipeline class for retail data processing."""
    
    def __init__(self, app_name):
        """Initialize the ETL pipeline with Spark session."""
        self.app_name = app_name
        self.spark = None
        self.sc = None
        self.data = None
        self.cleaned_data = None
        self.parsed_data = None
        
    def initialize_spark(self):
        """Initialize Spark session and context."""
        try:
            logger.info("Initializing Spark session...")
            self.spark = (SparkSession.builder
                         .master('local')
                         .appName(self.app_name)
                         .getOrCreate())
            
            self.sc = self.spark.sparkContext
            logger.info("Spark session initialized successfully")
            
        except Exception as e:
            logger.error(f"Failed to initialize Spark: {str(e)}")
            raise
    
    def extract_data(self, file_path: str):
        """Extract data from CSV file."""
        try:
            logger.info(f"Extracting data from {file_path}...")
            
            # Read CSV with proper schema inference
            df = self.spark.read.csv(file_path, header=True, inferSchema=True)
            self.data = df.rdd
            
            logger.info(f"Data extracted successfully. Total records: {self.data.count()}")
            
        except Exception as e:
            logger.error(f"Failed to extract data: {str(e)}")
            raise
    
    def clean_data(self):
        """Clean and prepare data for analysis."""
        try:
            logger.info("Starting data cleaning process...")
            
            # Get column names
            columns = self.data.toDF().columns
            
            # Replace missing CustomerID values with "Unknown"
            self.cleaned_data = self.data.map(lambda row: Row(**{
                col: ("Unknown" if col == "CustomerID" and row[col] is None else row[col])
                for col in columns
            }))
            
            # Verify cleaning
            null_customers = self.cleaned_data.filter(lambda row: row['CustomerID'] is None).count()
            if null_customers == 0:
                logger.info("Data cleaning completed successfully")
            else:
                logger.warning(f"Still found {null_customers} null CustomerID values")
                
        except Exception as e:
            logger.error(f"Failed to clean data: {str(e)}")
            raise
    
    def parse_dates(self):
        """Parse date fields for time-based analysis."""
        try:
            logger.info("Parsing date fields...")
            
            self.parsed_data = self.cleaned_data.map(lambda row: Row(
                InvoiceNo=row['InvoiceNo'],
                StockCode=row['StockCode'],
                Description=row['Description'],
                Quantity=row['Quantity'],
                InvoiceDate=datetime.strptime(row['InvoiceDate'], "%m/%d/%Y %H:%M"),
                UnitPrice=row['UnitPrice'],
                CustomerID=row['CustomerID'],
                Country=row['Country']
            ))
            
            logger.info("Date parsing completed successfully")
            
        except Exception as e:
            logger.error(f"Failed to parse dates: {str(e)}")
            raise
    
    def analyze_sales_overview(self):
        """Analyze overall sales metrics."""
        try:
            logger.info("Analyzing sales overview...")
            
            # Total transactions
            total_transactions = self.cleaned_data.map(lambda row: row['InvoiceNo']).distinct().count()
            
            # Total quantity sold
            total_quantity = self.cleaned_data.map(lambda row: row['Quantity']).sum()
            
            # Total revenue
            total_revenue = self.cleaned_data.map(lambda row: row['UnitPrice'] * row['Quantity']).sum()
            
            # Average order value
            avg_order_value = total_revenue / total_transactions if total_transactions > 0 else 0
            
            results = {
                'total_transactions': total_transactions,
                'total_quantity': total_quantity,
                'total_revenue': round(total_revenue, 2),
                'avg_order_value': round(avg_order_value, 2)
            }
            
            logger.info("Sales overview analysis completed")
            return results
            
        except Exception as e:
            logger.error(f"Failed to analyze sales overview: {str(e)}")
            raise
    
    def analyze_products(self):
        """Analyze product performance metrics."""
        try:
            logger.info("Analyzing product performance...")
            
            # Top products by quantity
            top_products_qty = (self.cleaned_data
                               .map(lambda row: (row['Description'], row['Quantity']))
                               .reduceByKey(lambda a, b: a + b)
                               .sortBy(lambda x: x[1], ascending=False)
                               .take(10))
            
            # Top products by revenue
            top_products_rev = (self.cleaned_data
                               .map(lambda row: (row['Description'], row['Quantity'] * row['UnitPrice']))
                               .reduceByKey(lambda a, b: round(a + b, 2))
                               .sortBy(lambda x: x[1], ascending=False)
                               .take(10))
            
            logger.info("Product analysis completed")
            return top_products_qty, top_products_rev
            
        except Exception as e:
            logger.error(f"Failed to analyze products: {str(e)}")
            raise
    
    def analyze_customers(self):
        """Analyze customer behavior metrics."""
        try:
            logger.info("Analyzing customer behavior...")
            
            # Filter out unknown customers
            known_customers = self.cleaned_data.filter(lambda row: row['CustomerID'] != "Unknown")
            
            # Top spending customers
            top_spending = (known_customers
                           .map(lambda row: (row['CustomerID'], row['Quantity'] * row['UnitPrice']))
                           .reduceByKey(lambda a, b: round(a + b, 2))
                           .sortBy(lambda x: x[1], ascending=False)
                           .take(10))
            
            # Top customers by purchase frequency
            top_frequency = (known_customers
                            .map(lambda row: (row['CustomerID'], 1))
                            .reduceByKey(lambda a, b: a + b)
                            .sortBy(lambda x: x[1], ascending=False)
                            .take(10))
            
            # Average order value per customer
            avg_order_per_customer = (known_customers
                                     .map(lambda row: (row['CustomerID'], (row['Quantity'] * row['UnitPrice'], 1)))
                                     .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                                     .mapValues(lambda x: round(x[0] / x[1], 2))
                                     .sortBy(lambda x: x[1], ascending=False)
                                     .take(10))
            
            logger.info("Customer analysis completed")
            return top_spending, top_frequency, avg_order_per_customer
            
        except Exception as e:
            logger.error(f"Failed to analyze customers: {str(e)}")
            raise
    
    def analyze_time_patterns(self):
        """Analyze time-based patterns in sales."""
        try:
            logger.info("Analyzing time patterns...")
            
            if not self.parsed_data:
                raise ValueError("Parsed data not available. Run parse_dates() first.")
            
            # Sales by month
            sales_by_month = (self.parsed_data
                             .map(lambda row: ((row.InvoiceDate.year, row.InvoiceDate.month), 
                                              row.Quantity * row.UnitPrice))
                             .reduceByKey(lambda a, b: round(a + b, 2))
                             .sortByKey()
                             .collect())
            
            # Sales by day
            sales_by_day = (self.parsed_data
                           .map(lambda row: (row.InvoiceDate.date(), row.Quantity * row.UnitPrice))
                           .sortByKey()
                           .collect())
            
            # Sales by hour
            sales_by_hour = (self.parsed_data
                            .map(lambda row: (row.InvoiceDate.hour, row.Quantity * row.UnitPrice))
                            .reduceByKey(lambda a, b: round(a + b, 2))
                            .sortByKey()
                            .collect())
            
            logger.info("Time pattern analysis completed")
            return sales_by_month, sales_by_day, sales_by_hour
            
        except Exception as e:
            logger.error(f"Failed to analyze time patterns: {str(e)}")
            raise
    
    def analyze_geographic_patterns(self):
        """Analyze geographic distribution of sales."""
        try:
            logger.info("Analyzing geographic patterns...")
            
            if not self.parsed_data:
                raise ValueError("Parsed data not available. Run parse_dates() first.")
            
            # Sales by country
            sales_by_country = (self.parsed_data
                               .map(lambda row: (row.Country, row.Quantity * row.UnitPrice))
                               .reduceByKey(lambda a, b: round(a + b, 2))
                               .sortBy(lambda x: x[1], ascending=False)
                               .collect())
            
            # Transactions by country
            transactions_by_country = (self.parsed_data
                                     .map(lambda row: (row.Country, row.InvoiceNo))
                                     .distinct()
                                     .map(lambda x: (x[0], 1))
                                     .reduceByKey(lambda a, b: a + b)
                                     .sortBy(lambda x: x[1], ascending=False)
                                     .collect())
            
            logger.info("Geographic analysis completed")
            return sales_by_country, transactions_by_country
            
        except Exception as e:
            logger.error(f"Failed to analyze geographic patterns: {str(e)}")
            raise
    
    def generate_report(self, results):
        """Generate and save analysis report."""
        try:
            logger.info("Generating analysis report...")
            
            report = f"""
ONLINE RETAIL ANALYSIS REPORT
============================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

SALES OVERVIEW:
---------------
Total Transactions: {results['sales_overview']['total_transactions']:,}
Total Quantity Sold: {results['sales_overview']['total_quantity']:,}
Total Revenue: ${results['sales_overview']['total_revenue']:,.2f}
Average Order Value: ${results['sales_overview']['avg_order_value']:,.2f}

TOP PRODUCTS BY QUANTITY:
------------------------
"""
            
            for i, (product, qty) in enumerate(results['top_products_qty'], 1):
                report += f"{i}. {product}: {qty:,} units\n"
            
            report += f"""

TOP PRODUCTS BY REVENUE:
-----------------------
"""
            
            for i, (product, revenue) in enumerate(results['top_products_rev'], 1):
                report += f"{i}. {product}: ${revenue:,.2f}\n"
            
            report += f"""

TOP CUSTOMERS BY SPENDING:
-------------------------
"""
            
            for i, (customer_id, spending) in enumerate(results['top_spending_customers'], 1):
                report += f"{i}. Customer {customer_id}: ${spending:,.2f}\n"
            
            # Save report to file
            with open('retail_analysis_report.txt', 'w', encoding='utf-8') as f:
                f.write(report)
            
            logger.info("Report generated and saved to 'retail_analysis_report.txt'")
            
        except Exception as e:
            logger.error(f"Failed to generate report: {str(e)}")
            raise
    
    def run_pipeline(self, file_path: str = 'dataset/retail_dataset.csv'):
        """Run the complete ETL pipeline."""
        try:
            logger.info("Starting ETL pipeline...")
            
            # Initialize Spark
            self.initialize_spark()
            
            # Extract
            self.extract_data(file_path)
            
            # Transform
            self.clean_data()
            self.parse_dates()
            
            # Analyze
            sales_overview = self.analyze_sales_overview()
            top_products_qty, top_products_rev = self.analyze_products()
            top_spending, top_frequency, avg_order_per_customer = self.analyze_customers()
            sales_by_month, sales_by_day, sales_by_hour = self.analyze_time_patterns()
            sales_by_country, transactions_by_country = self.analyze_geographic_patterns()
            
            # Compile results
            results = {
                'sales_overview': sales_overview,
                'top_products_qty': top_products_qty,
                'top_products_rev': top_products_rev,
                'top_spending_customers': top_spending,
                'top_customers_frequency': top_frequency,
                'avg_order_per_customer': avg_order_per_customer,
                'sales_by_month': sales_by_month,
                'sales_by_day': sales_by_day,
                'sales_by_hour': sales_by_hour,
                'sales_by_country': sales_by_country,
                'transactions_by_country': transactions_by_country
            }
            
            # Generate report
            self.generate_report(results)
            
            logger.info("ETL pipeline completed successfully!")
            return results
            
        except Exception as e:
            logger.error(f"Pipeline failed: {str(e)}")
            raise
        
        finally:
            # Cleanup
            if self.spark:
                self.spark.stop()
                logger.info("Spark session stopped")


def main():
    """Main function to run the ETL pipeline."""
    try:
        # Create and run pipeline
        pipeline = RetailETLPipeline()
        results = pipeline.run_pipeline()
        
        # Print summary
        print("\n" + "="*50)
        print("ETL PIPELINE COMPLETED SUCCESSFULLY!")
        print("="*50)
        print(f"Total Revenue: ${results['sales_overview']['total_revenue']:,.2f}")
        print(f"Total Transactions: {results['sales_overview']['total_transactions']:,}")
        print(f"Report saved to: retail_analysis_report.txt")
        print("="*50)
        
    except Exception as e:
        logger.error(f"Pipeline execution failed: {str(e)}")
        sys.exit(1)


if __name__ == "__main__":
    main() 

2025-08-13 11:43:50,871 - INFO - Starting ETL pipeline...
2025-08-13 11:43:50,881 - INFO - Initializing Spark session...
2025-08-13 11:43:51,355 - INFO - Spark session initialized successfully
2025-08-13 11:43:51,357 - INFO - Extracting data from dataset/retail_dataset.csv...
2025-08-13 11:44:06,881 - INFO - Data extracted successfully. Total records: 541909
2025-08-13 11:44:06,888 - INFO - Starting data cleaning process...
2025-08-13 11:44:16,007 - INFO - Data cleaning completed successfully
2025-08-13 11:44:16,009 - INFO - Parsing date fields...
2025-08-13 11:44:16,011 - INFO - Date parsing completed successfully
2025-08-13 11:44:16,015 - INFO - Analyzing sales overview...
2025-08-13 11:44:46,367 - INFO - Sales overview analysis completed
2025-08-13 11:44:46,388 - INFO - Analyzing product performance...
2025-08-13 11:45:25,234 - INFO - Product analysis completed
2025-08-13 11:45:25,236 - INFO - Analyzing customer behavior...
2025-08-13 11:46:01,152 - INFO - Customer analysis complete